# Vhembe District Data Preprocessing & Analysis

## Installing Required Libraries

In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install plotly

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import re

## Combining the Dataset Sheets

In [0]:
file_path = "/Workspace/Users/hlophetshepiso4@gmail.com/Vhembe District Municipality Data Analysis/VDM - File for Close up Report.xlsx"

# Load the Excel workbook
excel_file = pd.ExcelFile(file_path)

# Display all sheet names
print(excel_file.sheet_names)

['7232 - March 2024', '7322 - Jun 2024', '7587 - Oct 2024', '7756 - Jan 2025', '8002 - Jun 2025', '8163 - Oct 2025', '8173 - Nov 2025', '8265 - Feb 2026', '8331 - Apr 2026', '0176 - Aug 2026', '0264 - Oct 2026']


In [0]:
# Sheets to exclude (early batches with incomplete or different data)
excluded_sheets = [
    "5776 - May 2022 Colli",
    "5776 - May 2022 Makha",
    "5776 - May 2022 – Thula",
    "5776 - May 2022 - Musin",
    "5776 - April 2023",
    "6779 - Feb Batch 1 2023",
    "6779 - Feb Batch 2 2023",
    "6779 - Apr Batch 1 2023",
    "6779 - Apr Batch 2 2023",
    "6779 - Jun Batch 2023"
]

all_data = []

for sheet in excel_file.sheet_names:
    if sheet in excluded_sheets:
        continue

    # Extract month and year from the sheet name
    match = re.search(r"([A-Za-z]+)\s+(\d{4})$", sheet.strip())

    if not match:
        print(f"Skipped: {sheet}")
        continue

    # Read all records from the sheet
    df = pd.read_excel(excel_file, sheet_name=sheet)

    # Add month and year to EVERY record
    df["Month"] = match.group(1)
    df["Year"] = int(match.group(2))
    df["Source_Sheet"] = sheet

    all_data.append(df)

# Combine all sheets into one dataset (ORIGINAL - no cleaning, all columns)
combined_df = pd.concat(all_data, ignore_index=True, sort=False)

print(f"Original combined_df: {len(combined_df):,} records, {len(combined_df.columns)} columns")
print(f"Sheets loaded: {len(all_data)}")

Original combined_df: 170,091 records, 94 columns
Sheets loaded: 11


## Preprocessing & Cleaning the Dataset

In [0]:
# Create a cleaned copy of the original combined_df for Data Studio / Looker Studio
# (Original combined_df with all columns is created in the cell above)
dashboard_df = combined_df.copy()

# ── Drop redundant columns to reduce file size below 100MB for Data Studio ──
# Phone numbers (9), emails (3), address lines (4), duplicate Municipal register (~28),
# and other unnecessary columns
cols_to_drop = [
    # Phone numbers
    "Bureau - Mobile Phone 1", "Bureau - Mobile Phone 2", "Bureau - Mobile Phone 3",
    "Bureau - Work Phone 1", "Bureau - Work Phone 2", "Bureau - Work Phone 3",
    "Bureau - Home Phone 1", "Bureau - Home Phone 2", "Bureau - Home Phone 3",
    # Email
    "Bureau - Has Email", "Bureau - Email 1", "Bureau - Email 2", "Bureau - Email 3",
    # Address lines (keep Postal Code)
    "Bureau - ResidentialAddress1Line1", "Bureau - ResidentialAddress1Line2",
    "Bureau - ResidentialAddress1Line3", "Bureau - ResidentialAddress1Line4",
    # Has Mobile Phone flag
    "Bureau - Has Mobile Phone",
    # Contact category
    "Bureau - Contact Category",
    # Linked spouse
    "Linked Spouse Application Record Key",
    # Middle name
    "Bureau - Middle Name",
    # Duplicate Municipal register columns
    "Municipal - Municipal Book", "Municipal - Account Number", "Municipal - Account Holder",
    "Municipal - IdNumber", "Municipal - Name", "Municipal - IDNumber",
    "Municipal - Town", "Municipal - MUNICIPALITY", "Municipal - TOWNSHIP_CODE",
    "Municipal - TOWNSHIP", "Municipal - ACCT_ID", "Municipal - BADGE_NBR",
    "Municipal - NAT_ID", "Municipal - NAME", "Municipal - ADDRESS1",
    "Municipal - ADDRESS2", "Municipal - ADDRESS3", "Municipal - TARIFF",
    "Municipal - AMP_LIMIT", "Municipal - FBE_TYPE", "Municipal - CLAIM_MONTH",
    "Municipal - Municipality", "Municipal - Township", "Municipal - AcountNo",
    "Municipal - IDNo", "Municipal - ApplicantName", "Municipal - Sheet",
    "Municipal - AppicantIdNumber", "Municipal - Notes"
]
cols_to_drop = [c for c in cols_to_drop if c in dashboard_df.columns]
dashboard_df = dashboard_df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} redundant columns — {len(dashboard_df.columns)} remaining")

# (Verification Risk Level is computed in the export cell, not here)

# ── Advanced cleaning for Data Studio / Looker Studio compatibility ──
# 1. Remove ALL control characters (ASCII 0-31 and 127) from every string column
# 2. Replace commas with spaces (Databricks display download does not quote fields,
#    so commas in values like R3 720 or MASHABA NYANISI break CSV parsing)
import re as _re

_ctrl_pattern = _re.compile('[' + ''.join(chr(i) for i in range(32)) + chr(127) + ']+')

for col in dashboard_df.columns:
    s = dashboard_df[col]

    if s.dtype == bool:
        s = s.astype("string")
    elif s.dtype == object:
        s = s.astype("string")
    else:
        continue

    # Replace all control characters with a single space
    s = s.str.replace(_ctrl_pattern, " ", regex=True)

    # Replace commas with spaces to prevent CSV misparsing in Data Studio
    s = s.str.replace(",", " ", regex=False)

    # Collapse multiple spaces into one
    s = s.str.replace(r' {2,}', ' ', regex=True)

    # Strip leading / trailing whitespace
    s = s.str.strip()

    dashboard_df[col] = s

# ── Fill NULL values: 0 for numeric columns, "Not Filled" for string columns ──
for col in dashboard_df.columns:
    if dashboard_df[col].dtype in ["int64", "float64", "long", "double"]:
        dashboard_df[col] = dashboard_df[col].fillna(0)
    elif dashboard_df[col].dtype == "object" or dashboard_df[col].dtype == "string":
        dashboard_df[col] = dashboard_df[col].fillna("Not Filled")

print(f"Filled NULLs across {len(dashboard_df.columns)} columns")

# ── Shorten Rule column values to just Passed/Failed to reduce CSV size ──
# Full descriptions like "Passed - Applicant Bureau Income Less than R3 720" take ~50 chars
# per row; shortening to "Passed"/"Failed" saves ~60 MB across all Rule columns
rule_cols = [c for c in dashboard_df.columns if c.startswith("Rule -")]
for col in rule_cols:
    dashboard_df[col] = (
        dashboard_df[col]
        .astype("string")
        .str.split(" - ")
        .str[0]
        .str.strip()
    )

# Also shorten EnrichmentOutcome (e.g. "01 Enrichment Passed" -> "Passed")
dashboard_df["Municipal - EnrichmentOutcome"] = (
    dashboard_df["Municipal - EnrichmentOutcome"]
    .astype("string")
    .str.split(" - ")
    .str[-1]
    .str.strip()
)

# ── Drop derived bucket columns (raw numeric columns already retained) ──
bucket_cols = [
    "Bureau - IncomeBucket", "Bureau - DebtBucket",
    "Bureau - Directorship Bucket", "Bureau - Properties Bucket",
    "Bureau - Age Group"
]
bucket_cols = [c for c in bucket_cols if c in dashboard_df.columns]
dashboard_df = dashboard_df.drop(columns=bucket_cols)
print(f"Dropped {len(bucket_cols)} derived bucket columns — {len(dashboard_df.columns)} remaining")

# ── Drop non-essential columns for Data Studio dashboard ──
# Keep only columns needed for visualizations: outcomes, demographics, geography,
# income/debt, directorships, properties, employer, time, and key identifiers
dashboard_drop = [
    "Municipal - Book", "Municipal - PhoneNumber", "Municipal - EnrichmentOutcome",
    "Bureau - IDN", "Bureau - First Name", "Bureau - Surname",
    "Bureau - Has Name", "Bureau - Deceased Status",
    "Bureau - Deceased Date", "Source_Sheet"
]
dashboard_drop = [c for c in dashboard_drop if c in dashboard_df.columns]
dashboard_df = dashboard_df.drop(columns=dashboard_drop)
print(f"Dropped {len(dashboard_drop)} non-dashboard columns — {len(dashboard_df.columns)} remaining")

print(f"Cleaned {len(dashboard_df.columns)} columns across {len(dashboard_df):,} records")

# Display every record and all columns
# display(dashboard_df)

# Confirm that Month and Year are populated
print("Total records:", len(dashboard_df))
print("Missing Month values:", dashboard_df["Month"].isna().sum())
print("Missing Year values:", dashboard_df["Year"].isna().sum())

Dropped 50 redundant columns — 44 remaining
Filled NULLs across 44 columns
Dropped 5 derived bucket columns — 39 remaining
Dropped 10 non-dashboard columns — 29 remaining
Cleaned 29 columns across 170,091 records
Total records: 170091
Missing Month values: 0
Missing Year values: 0


## Records Per Province

In [0]:
# Replace null and blank province values with "Not Filled"
combined_df["Bureau - Province"] = (
    combined_df["Bureau - Province"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Not Filled")
)

# Count records by province
province_counts = combined_df["Bureau - Province"].value_counts()

province_summary = province_counts.reset_index()
province_summary.columns = ["Province", "Total Records"]

# Calculate percentages
total_records = province_summary["Total Records"].sum()

province_summary["Percentage"] = (
    province_summary["Total Records"]
    .div(total_records)
    .mul(100)
    .map(lambda x: f"{x:.2f}%")
)

display(province_summary)

Province,Total Records,Percentage
Limpopo,143687,84.48%
Not Filled,14325,8.42%
Gauteng,10242,6.02%
North West,875,0.51%
Mpumalanga,490,0.29%
Free State,123,0.07%
Western Cape,105,0.06%
KwaZulu-Natal,101,0.06%
Eastern Cape,92,0.05%
Northern Cape,49,0.03%


## Getting all the Column names

In [0]:
# Getting all the column names
for col in combined_df.columns:
    print(col)

Municipal - Record Key
Municipal - Book
Municipal - AccountNumber
Municipal - IDN
Municipal - Firstname
Municipal - Surname
Municipal - PhoneNumber
Municipal - EnrichmentOutcome
Linked Spouse Application Record Key
Rule - Multiple Accounts
Bureau - IDN
Rule - Valid ID Verification
Bureau - First Name
Bureau - Middle Name
Bureau - Surname
Bureau - Has Name
Bureau - Age
Bureau - Age Group
Rule - Child Headed Household
Bureau - Gender
Bureau - Population Group
Bureau - Municipality
Rule - Municipality Check
Bureau - Province
Bureau - Deceased Status
Rule - Deceased Status
Bureau - Income
Bureau - IncomeBucket
Rule - Monthly Income
Bureau - Debt
Bureau - DebtBucket
Rule - Monthly Debt
Bureau - Number of Active Companies
Bureau - Directorship Bucket
Rule - Directorship Status
Bureau - Number Of Properties
Bureau - Properties Bucket
Rule - Number Of Properties
Rule - Home Ownership Status
Bureau - Contact Category
Bureau - Employer
Bureau - Has Mobile Phone
Bureau - Mobile Phone 1
Bureau - M

# 1. Verification Outcomes – Three-Year Analysis



### A) Records per Municipality

In [0]:
target_municipalities = [
    "Makhado",
    "Musina",
    "Thulamela",
    "Collins Chabane"
]

municipality_counts = (
    combined_df["Bureau - Municipality"]
    .astype("string")
    .str.strip()
    .value_counts(dropna=False)
)

# Keep values matching our four municipalities
filtered_counts = municipality_counts[
    municipality_counts.index.str.contains(
        "|".join(target_municipalities),
        case=False,
        na=False,
        regex=True
    )
]

print("Record counts for our four municipalities:")
print(filtered_counts)

Record counts for our four municipalities:
Bureau - Municipality
LIM345 - Collins Chabane    77124
LIM344 - Makhado            21970
LIM343 - Thulamela          21762
LIM341 - Musina             20539
Name: count, dtype: Int64


-  **Collins Chabane** Municipality has the highest number of records within Vhembe District, at 77 124
- **Makhado** Local Municipality has the second highest number of records within Vhembe District, at 21 970
- **Thulamela** Local Municipality has the third highest number of records within Vhembe District, at 21 762
- **Musina** Local Municipality has the lowest number of records within Vhembe District, at 20 539

## B) Indigent Application Outcomes

In [0]:
# Check indigent application outcomes
print("\nIndigent Application Outcomes:")
print(combined_df["Rule - Indigent Application"].value_counts(dropna=False))


Indigent Application Outcomes:
Rule - Indigent Application
Passed - Indigent Application Successful      99340
Failed - Indigent Application Unsuccessful    70751
Name: count, dtype: int64


- 99 340 applications were **approved** as Indigent
- 70 751 applications were **declined** to be indigent

## C) Checking Deceased Status

In [0]:
# Check deceased status
print("\nDeceased Status:")
print(combined_df["Rule - Deceased Status"].value_counts(dropna=False))


Deceased Status:
Rule - Deceased Status
Passed - Applicant Alive       157529
Failed - Applicant Deceased     12562
Name: count, dtype: int64


- 157 529 applicants are still alive
- 12 562 applicants have deceased

## D) Child-headed household status

In [0]:
# Check child-headed household status
print("\nChild-Headed Household Status:")
print(combined_df["Rule - Child Headed Household"].value_counts(dropna=False))


Child-Headed Household Status:
Rule - Child Headed Household
Adult Applicant    170055
Child Applicant        36
Name: count, dtype: int64


Only 36 households are child-headed households

In [0]:
# Output child-headed household records
child_headed_households = combined_df.loc[
    combined_df["Rule - Child Headed Household"] == "Child Applicant"
]

display(child_headed_households)

Municipal - Record Key,Municipal - Book,Municipal - AccountNumber,Municipal - IDN,Municipal - Firstname,Municipal - Surname,Municipal - PhoneNumber,Municipal - EnrichmentOutcome,Linked Spouse Application Record Key,Rule - Multiple Accounts,Bureau - IDN,Rule - Valid ID Verification,Bureau - First Name,Bureau - Middle Name,Bureau - Surname,Bureau - Has Name,Bureau - Age,Bureau - Age Group,Rule - Child Headed Household,Bureau - Gender,Bureau - Population Group,Bureau - Municipality,Rule - Municipality Check,Bureau - Province,Bureau - Deceased Status,Rule - Deceased Status,Bureau - Income,Bureau - IncomeBucket,Rule - Monthly Income,Bureau - Debt,Bureau - DebtBucket,Rule - Monthly Debt,Bureau - Number of Active Companies,Bureau - Directorship Bucket,Rule - Directorship Status,Bureau - Number Of Properties,Bureau - Properties Bucket,Rule - Number Of Properties,Rule - Home Ownership Status,Bureau - Contact Category,Bureau - Employer,Bureau - Has Mobile Phone,Bureau - Mobile Phone 1,Bureau - Mobile Phone 2,Bureau - Mobile Phone 3,Bureau - Work Phone 1,Bureau - Work Phone 2,Bureau - Work Phone 3,Bureau - Home Phone 1,Bureau - Home Phone 2,Bureau - Home Phone 3,Bureau - Has Email,Bureau - Email 1,Bureau - Email 2,Bureau - Email 3,Bureau - ResidentialAddress1Line1,Bureau - ResidentialAddress1Line2,Bureau - ResidentialAddress1Line3,Bureau - ResidentialAddress1Line4,Bureau - ResidentialAddress1PostalCode,Rule - Indigent Application,Month,Year,Source_Sheet,Bureau - Deceased Date,Municipal - Municipal Book,Municipal - Account Number,Municipal - Account Holder,Municipal - IdNumber,Municipal - Name,Municipal - IDNumber,Municipal - Town,Municipal - MUNICIPALITY,Municipal - TOWNSHIP_CODE,Municipal - TOWNSHIP,Municipal - ACCT_ID,Municipal - BADGE_NBR,Municipal - NAT_ID,Municipal - NAME,Municipal - ADDRESS1,Municipal - ADDRESS2,Municipal - ADDRESS3,Municipal - TARIFF,Municipal - AMP_LIMIT,Municipal - FBE_TYPE,Municipal - CLAIM_MONTH,Municipal - Municipality,Municipal - Township,Municipal - AcountNo,Municipal - IDNo,Municipal - ApplicantName,Municipal - Sheet,Municipal - AppicantIdNumber,Municipal - Notes
2546,Collins Chabane,9384417426,5601018948083,MARINGA,MARINGA,null,01 Enrichment Passed,null,Passed - Applicant Has A Single Indigent Account,5.601018948083E12,Passed - Valid ID Number Captured,Risimati,Derrick,Maringa,true,-32.0,06. Sixties,Child Applicant,Male,Unknown,LIM345 - Collins Chabane,Passed - Applicant Lives In Municipality,Limpopo,false,Passed - Applicant Alive,0.0,01. R0 - R2 500,"Passed - Applicant Bureau Income Less than R3,720",0.0,01. R0 - R2 500,"Passed - Applicant Bureau Debt Instalments Less than R3,720",null,00. 0 Companies,Passed - Applicant Is Not A Director,0.0,00. 0 Properties,Passed - Applicant Does Not Own Multiple Properties,Passed - Applicant Is A Home Owner,05 Very Low,null,false,null,null,null,null,null,null,null,null,null,false,null,null,null,null,null,null,null,null,Passed - Indigent Application Successful,March,2024,7232 - March 2024,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
7757,Thulamela 2,342438,6711267166901,null,MAWELA VHONANI SYLVIA,null,01 Enrichment Passed,null,Passed - Applicant Has A Single Indigent Account,6.711267166086E12,Passed - Valid ID Number Captured,null,null,null,false,-43.0,05. Fifties,Child Applicant,Male,Unknown,null,Passed - Applicant Lives In Municipality,Not Filled,false,Passed - Applicant Alive,0.0,01. R0 - R2 500,"Passed - Applicant Bureau Income Less than R3,720",0.0,01. R0 - R2 500,"Passed - Applicant Bureau Debt Instalments Less than R3,720",null,00. 0 Companies,Passed - Applicant Is Not A Director,0.0,00. 0 Properties,Passed - Applicant Does Not Own Multiple Properties,Passed - Applicant Is A Home Owner,05 Very Low,null,false,null,null,null,null,null,null,null,null,null,false,null,null,null,null,null,null,null,null,Passed - Indigent Application Successful,March,2024,7232 - Mar

## E) Applicants with Multiple Accounts

In [0]:
# Check duplicate indicators
print("\nMultiple Accounts:")
print(combined_df["Rule - Multiple Accounts"].value_counts(dropna=False))


Multiple Accounts:
Rule - Multiple Accounts
Passed - Applicant Has A Single Indigent Account     116998
Failed - Applicant Has Multiple Indigent Accounts     52365
Passed - Applicant Has Multiple Indigent Accounts       728
Name: count, dtype: int64


## F) Indigent Outcome Count Per Year

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Convert your combined pandas dataset to a Spark DataFrame
pdf = combined_df.copy()

for c in pdf.select_dtypes(include="object").columns:
    pdf[c] = pdf[c].astype("string")

spark_df = spark.createDataFrame(pdf)

# Define a window for each year
year_window = Window.partitionBy("Year")

# Count records by year and outcome, then calculate percentages
yearly_outcomes = (
    spark_df
    .filter(F.col("Year").isin(2024, 2025, 2026))
    .groupBy("Year", "Rule - Indigent Application")
    .agg(F.count("*").alias("Total Records"))
    .withColumn(
        "Percentage",
        F.concat(
            F.round(
                F.col("Total Records") /
                F.sum("Total Records").over(year_window) * 100,
                2
            ).cast("string"),
            F.lit("%")
        )
    )
    .orderBy("Year", "Rule - Indigent Application")
)

display(yearly_outcomes)

Year,Rule - Indigent Application,Total Records,Percentage
2024,Failed - Indigent Application Unsuccessful,20259,32.56%
2024,Passed - Indigent Application Successful,41963,67.44%
2025,Failed - Indigent Application Unsuccessful,39756,64.4%
2025,Passed - Indigent Application Successful,21976,35.6%
2026,Failed - Indigent Application Unsuccessful,10736,23.27%
2026,Passed - Indigent Application Successful,35401,76.73%


## G) Indigent Outcome Count For Each Municipality

In [0]:
df = combined_df.copy()

# Clean municipality names
df["Municipality"] = (
    df["Bureau - Municipality"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Filter using partial name matching
municipality_patterns = {
    "Makhado Local Municipality": "makhado",
    "Musina Local Municipality": "musina",
    "Thulamela Local Municipality": "thulamela",
    "Collins Chabane Local Municipality": "collins chabane"
}

df["Municipality"] = df["Municipality"].apply(
    lambda x: next(
        (name for name, pattern in municipality_patterns.items()
         if pd.notna(x) and pattern in x),
        None
    )
)

df = df[df["Municipality"].notna()]

# Count outcomes for each municipality
municipality_by_outcome = pd.crosstab(
    df["Municipality"],
    df["Rule - Indigent Application"]
).reindex(list(municipality_patterns.keys()), fill_value=0)

# Calculate percentages within each municipality
municipality_percentages = (
    municipality_by_outcome
    .div(municipality_by_outcome.sum(axis=1).replace(0, pd.NA), axis=0)
    .mul(100)
    .round(2)
)

# Add % symbol to percentage values
municipality_percentages = municipality_percentages.map(
    lambda x: f"{x:.2f}%" if pd.notna(x) else "0.00%"
)

# Combine counts and percentages into the same output
municipality_output = municipality_by_outcome.astype(str) + " (" + municipality_percentages + ")"

display(municipality_output.reset_index())

Municipality,Failed - Indigent Application Unsuccessful,Passed - Indigent Application Successful
Makhado Local Municipality,3390 (15.43%),18580 (84.57%)
Musina Local Municipality,8328 (40.55%),12211 (59.45%)
Thulamela Local Municipality,11074 (50.89%),10688 (49.11%)
Collins Chabane Local Municipality,25422 (32.96%),51702 (67.04%)


## H) Indigent Outcome Count For Each Year & Municipality

In [0]:
df = combined_df.copy()

# Standardise municipality names
mun = df["Bureau - Municipality"].astype("string").str.strip().str.casefold()

df["Municipality"] = pd.NA
municipality_map = {
    "makhado": "Makhado Local Municipality",
    "musina": "Musina Local Municipality",
    "thulamela": "Thulamela Local Municipality",
    "collins chabane": "Collins Chabane Local Municipality"
}

for pattern, name in municipality_map.items():
    df.loc[mun.str.contains(pattern, na=False), "Municipality"] = name

# Keep the three reporting years and recognised municipalities
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df = df[
    df["Year"].isin([2024, 2025, 2026]) &
    df["Municipality"].notna()
].copy()

# FIRST TABLE: verification outcomes by year and municipality
first_table = (
    pd.crosstab(
        [df["Year"], df["Municipality"]],
        df["Rule - Indigent Application"]
    )
    .reindex(
        pd.MultiIndex.from_product(
            [[2024, 2025, 2026], municipality_map.values()],
            names=["Year", "Municipality"]
        ),
        fill_value=0
    )
    .reset_index()
)

display(first_table)

Year,Municipality,Failed - Indigent Application Unsuccessful,Passed - Indigent Application Successful
2024,Makhado Local Municipality,890,4540
2024,Musina Local Municipality,3262,4812
2024,Thulamela Local Municipality,1686,3620
2024,Collins Chabane Local Municipality,8065,26250
2025,Makhado Local Municipality,1728,3901
2025,Musina Local Municipality,2055,2454
2025,Thulamela Local Municipality,8476,4124
2025,Collins Chabane Local Municipality,14657,9995
2026,Makhado Local Municipality,772,10139
2026,Musina Local Municipality,3011,4945


## I) Checking for duplicate rows

In [0]:
df = combined_df.copy()

# Count rows that are exact duplicates of an earlier row
duplicate_mask = df.duplicated(keep="first")

print("Total records:", len(df))
print("Duplicate rows:", duplicate_mask.sum())
print("Unique records:", len(df) - duplicate_mask.sum())

# Display duplicate examples
duplicate_records = df[df.duplicated(keep=False)]

if not duplicate_records.empty:
    display(duplicate_records.head(50))

Total records: 170091
Duplicate rows: 0
Unique records: 170091


There are zero duplicated rows in the dataset

# 2. Applicant Segmentation and Risk Analysis



## A)  Distribution of applicants by gender across all four municipalities.

In [0]:
df = combined_df.copy()

# Standardise municipality names
mun = df["Bureau - Municipality"].astype("string").str.strip().str.casefold()

df["Municipality"] = pd.NA
municipality_map = {
    "makhado": "Makhado",
    "musina": "Musina",
    "thulamela": "Thulamela",
    "collins chabane": "Collins Chabane"
}

for pattern, name in municipality_map.items():
    df.loc[mun.str.contains(pattern, na=False), "Municipality"] = name

df = df[df["Municipality"].notna()].copy()

# Clean gender values
df["Gender"] = (
    df["Bureau - Gender"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Not recorded")
)

# Count applicants by municipality and gender
gender_table = pd.crosstab(
    df["Municipality"],
    df["Gender"],
    margins=True,
    margins_name="Total"
)

display(gender_table.reset_index())

# Calculate gender percentages within each municipality
gender_counts = pd.crosstab(df["Municipality"], df["Gender"])

gender_percentages = (
    gender_counts
    .div(gender_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
    .map(lambda x: f"{x:.2f}%")
)

display(gender_percentages.reset_index())

Municipality,Female,Male,Total
Collins Chabane,47122,30002,77124
Makhado,14546,7424,21970
Musina,14596,5943,20539
Thulamela,12267,9495,21762
Total,88531,52864,141395


Municipality,Female,Male
Collins Chabane,61.10%,38.90%
Makhado,66.21%,33.79%
Musina,71.06%,28.94%
Thulamela,56.37%,43.63%


A majority of our applicants are female across all four municipalities

## B) Records Per Income Bucket

In [0]:
# Check available income buckets and their record counts
income_summary = (
    combined_df["Bureau - IncomeBucket"]
    .fillna("Not recorded")
    .astype(str)
    .str.strip()
    .replace("", "Not recorded")
    .value_counts()
    .rename_axis("Income Bucket")
    .reset_index(name="Total Applicants")
)

income_summary["Percentage"] = (
    income_summary["Total Applicants"]
    / len(combined_df) * 100
).round(2).map(lambda x: f"{x:.2f}%")

display(income_summary)

Income Bucket,Total Applicants,Percentage
01. R0 - R2 500,83143,48.88%
00. Unknown,40151,23.61%
02. R2 500 - R5 000,28778,16.92%
03. R5 000 - R10 000,12117,7.12%
04. R10 000 - R20 000,3884,2.28%
05. R20 000 - R30 000,1108,0.65%
06. R30 000 - R40 000,508,0.30%
07. R40 000 +,402,0.24%


## C) Applicants linked to registered businesses

In [0]:
df = combined_df.copy()

municipalities = [
    "LIM341 - Musina",
    "LIM343 - Thulamela",
    "LIM344 - Makhado",
    "LIM345 - Collins Chabane"
]

directorship_accounts = df.loc[
    df["Bureau - Municipality"].isin(municipalities)
    & pd.to_numeric(
        df["Bureau - Number of Active Companies"],
        errors="coerce"
    ).ge(1),
    [
        "Bureau - Surname",
        "Bureau - Municipality",
        "Bureau - Directorship Bucket",
        "Rule - Indigent Application"
    ]
].drop_duplicates()

# display(directorship_accounts)

## D) Records Per Directorship Bucket

In [0]:
df = combined_df.copy()

# Clean directorship categories
df["Directorship Bucket"] = (
    df["Bureau - Directorship Bucket"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Not recorded")
)

# Count and calculate percentages
directorship_summary = (
    df["Directorship Bucket"]
    .value_counts(dropna=False)
    .rename_axis("Directorship Category")
    .reset_index(name="Applicants")
)

directorship_summary["Percentage"] = (
    directorship_summary["Applicants"]
    / len(df) * 100
).round(2).map(lambda x: f"{x:.2f}%")

display(directorship_summary)

Directorship Category,Applicants,Percentage
00. 0 Companies,160610,94.43%
01. 1 Companies,7450,4.38%
02. 2 Companies,1492,0.88%
03. 3-5 Companies,481,0.28%
04. 6-10 Companies,48,0.03%
05. 10+ Properties,10,0.01%


## E) Applicants who own Property

In [0]:
# Getting Accounts that own Properties
property_accounts = df.loc[
    df["Bureau - Municipality"].isin(municipalities)
    & pd.to_numeric(
        df["Bureau - Number Of Properties"],
        errors="coerce"
    ).ge(1),
    [
        "Bureau - Surname",
        "Bureau - Municipality",
        "Bureau - Number Of Properties",
        "Rule - Indigent Application"
    ]
].drop_duplicates()

# display(property_accounts)

## F) Records per Property Ownership

In [0]:
df = combined_df.copy()

# Convert property values to numeric
df["Property Count"] = pd.to_numeric(
    df["Bureau - Number Of Properties"],
    errors="coerce"
)

# Group property ownership into categories
df["Property Category"] = df["Property Count"].apply(
    lambda x: "Not recorded" if pd.isna(x)
    else "No properties" if x == 0
    else "1 property" if x == 1
    else "More than 1 property"
)

# Count records and calculate percentages
property_summary = (
    df["Property Category"]
    .value_counts()
    .rename_axis("Property Category")
    .reset_index(name="Applicants")
)

property_summary["Percentage"] = (
    property_summary["Applicants"] / len(df) * 100
).round(2).map(lambda x: f"{x:.2f}%")

display(property_summary)

Property Category,Applicants,Percentage
No properties,79412,46.69%
Not recorded,61742,36.30%
1 property,23046,13.55%
More than 1 property,5891,3.46%


## G)  Identification of applicants employed by government departments or public institutions

In [0]:
# Filter to the four Vhembe municipalities
municipalities = [
    "LIM341 - Musina",
    "LIM343 - Thulamela",
    "LIM344 - Makhado",
    "LIM345 - Collins Chabane"
]

# Keywords for preliminary identification of government/public employers
government_pattern = (
    r"municipality|municipal|government|department|"
    r"provincial|national|public service|"
    r"south african police service|\bSAPS\b|"
    r"\bSANDF\b|public hospital|public school|"
    r"department of education|department of health|"
    r"national park|national health laboratory|"
    r"local government|provincial admin|"
    r"national treasury|public works|water affairs"
)

# Exclude pension and grant-related descriptions
exclude_pattern = (
    r"pension|pensioner|pension fund|"
    r"\bgrant\b|SASSA|social security|"
    r"old age|disability grant|child support"
)

employer = df["Bureau - Employer"].astype("string").str.strip()

# Identify potential public-sector employers
government_mask = employer.str.contains(
    government_pattern,
    case=False,
    na=False,
    regex=True
)

# Identify pension/grant-related descriptions
exclude_mask = employer.str.contains(
    exclude_pattern,
    case=False,
    na=False,
    regex=True
)

# Apply municipality and employer filters
mask = (
    df["Bureau - Municipality"].isin(municipalities)
    & government_mask
    & ~exclude_mask
)

government_accounts = df.loc[
    mask,
    [
        "Municipal - AccountNumber",
        "Bureau - First Name",
        "Bureau - Surname",
        "Bureau - Municipality",
        "Bureau - Employer"
    ]
].drop_duplicates()

# display(government_accounts)

## H) Records Per Government/Public Institution-employed applicants

In [0]:
df = combined_df.copy()

employer_col = "Bureau - Employer"

# Clean employer descriptions
df[employer_col] = (
    df[employer_col]
    .astype("string")
    .str.strip()
)

employer_clean = df[employer_col].str.casefold()

# Exclude pension and grant-related descriptions
exclude_pattern = (
    r"pension|pensioner|pension fund|"
    r"\bgrant\b|sassa|social security|"
    r"old age|disability grant|child support"
)

excluded = employer_clean.str.contains(
    exclude_pattern,
    regex=True,
    na=False
)

# Identify likely government departments and public institutions
include_pattern = (
    r"department|municipality|municipal|"
    r"\bgovernment\b|provincial|"
    r"\bministry\b|national park|"
    r"national health laboratory|"
    r"public hospital|public school|"
    r"local government|provincial admin|"
    r"public works|water affairs|"
    r"national treasury|national biodiversity institute"
)

likely_public_employer = employer_clean.str.contains(
    include_pattern,
    regex=True,
    na=False
)

# Flag generic or ambiguous descriptions for manual review
ambiguous_pattern = (
    r"^government$|government social|"
    r"^national food$|^national trading$"
)

ambiguous = employer_clean.str.contains(
    ambiguous_pattern,
    regex=True,
    na=False
)

# Retain likely public-sector employers, excluding grants and pensions
government_employees = df[
    ~excluded
    & likely_public_employer
    & ~ambiguous
    & df[employer_col].notna()
].copy()

# Summarise by employer
employer_summary = (
    government_employees
    .groupby(employer_col)
    .size()
    .reset_index(name="Total Records")
    .rename(columns={employer_col: "Employer"})
    .sort_values("Total Records", ascending=False)
)

# Calculate percentage among retained records
total_records = employer_summary["Total Records"].sum()

employer_summary["Percentage"] = (
    employer_summary["Total Records"]
    .div(total_records if total_records else 1)
    .mul(100)
    .map(lambda x: f"{x:.2f}%")
)

# display(employer_summary)

## I) Adding a Verification Risk Level Column

In [0]:
import numpy as np

# ── Verification Risk Level column on ALL combined_df records ──
#
# Priority: Not Approved > Unassessed > High > Medium > Low
# (Verification risk only applies to approved applications.)

# ── Unassessed — Insufficient information ──
# Missing, ambiguous or unavailable information prevents a reliable assessment.
# Triggered when bureau enrichment failed or no identity data was returned.
unassessed_mask = (
    combined_df["Municipal - EnrichmentOutcome"]
        .astype("string")
        .str.contains("Failed", case=False, na=False)
    | combined_df["Bureau - Has Name"].isna()
    | (combined_df["Bureau - Has Name"] == False)
)

# ── High — Urgent verification review ──
# Confirmed identity mismatch, deceased-status conflict, or duplicate accounts
# suggesting possible improper duplication.
high_mask = (
    combined_df["Rule - Valid ID Verification"].astype("string").str.startswith("Failed", na=False)
    | combined_df["Rule - Deceased Status"].astype("string").str.startswith("Failed", na=False)
    | combined_df["Rule - Multiple Accounts"].astype("string").str.startswith("Failed", na=False)
)

# ── Medium — Further assessment required ──
# Unconfirmed municipality, incomplete address, multiple accounts requiring
# reconciliation, income / property / directorship information requiring
# policy assessment, or employment by a government department / public institution.
_gov_include = (
    r"department|municipality|municipal|\bgovernment\b|provincial|"
    r"\bministry\b|national park|national health laboratory|"
    r"public hospital|public school|local government|"
    r"provincial admin|public works|water affairs|"
    r"national treasury|national biodiversity institute"
)
_gov_exclude = (
    r"pension|pensioner|pension fund|\bgrant\b|sassa|"
    r"social security|old age|disability grant|child support"
)
_employer = combined_df["Bureau - Employer"].astype("string").str.strip().str.casefold()
_gov_employed = (
    _employer.str.contains(_gov_include, regex=True, na=False)
    & ~_employer.str.contains(_gov_exclude, regex=True, na=False)
)

medium_mask = (
    combined_df["Rule - Municipality Check"].astype("string").str.startswith("Failed", na=False)
    | combined_df["Bureau - ResidentialAddress1PostalCode"].isna()
    | combined_df["Rule - Multiple Accounts"].astype("string").str.contains("Passed - Applicant Has Multiple", na=False)
    | combined_df["Rule - Monthly Income"].astype("string").str.startswith("Failed", na=False)
    | combined_df["Rule - Number Of Properties"].astype("string").str.startswith("Failed", na=False)
    | pd.to_numeric(combined_df["Bureau - Number Of Properties"], errors="coerce").ge(1)
    | pd.to_numeric(combined_df["Bureau - Number of Active Companies"], errors="coerce").ge(1)
    | combined_df["Rule - Directorship Status"].astype("string").str.startswith("Failed", na=False)
    | _gov_employed
)

# Only assess approved applications — declined applications are not actionable
approved_mask = (
    combined_df["Rule - Indigent Application"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .str.contains("pass", na=False)
)

# ── Classify ──
combined_df["Verification Risk Level"] = np.select(
    [~approved_mask, unassessed_mask, high_mask, medium_mask],
    [
        "Not Applicable — Application Not Approved",
        "Unassessed — Insufficient information",
        "High — Urgent verification review",
        "Medium — Further assessment required",
    ],
    default="Low — No selected risk flag identified",
)

# display(combined_df)

print("\n=== Verification Risk Level Distribution ===")
risk_dist = (
    combined_df["Verification Risk Level"]
    .value_counts()
    .rename_axis("Risk Level")
    .reset_index(name="Record Count")
)
risk_dist["Percentage"] = (
    risk_dist["Record Count"] / len(combined_df) * 100
).round(2).map(lambda x: f"{x:.2f}%")
# display(risk_dist)


=== Verification Risk Level Distribution ===


   
## J) Geographic Analysis and Heat Maps

Where residential address information is available, the report incorporates geographic heat maps showing the distribution and concentration of indigent applicants across the four Vhembe municipalities. The first heat map highlights areas with high concentrations of approved and declined applications. The second heat map shows the verification risk level breakdown for approved applicants.

In [0]:
import plotly.express as px

# ── Filter to Vhembe municipalities ──
municipalities = [
    "LIM341 - Musina",
    "LIM343 - Thulamela",
    "LIM344 - Makhado",
    "LIM345 - Collins Chabane"
]

df = combined_df[
    combined_df["Bureau - Municipality"].isin(municipalities)
].copy()

# Classify application outcome
outcome = df["Rule - Indigent Application"].astype("string").str.strip().str.casefold()
df["Map Category"] = np.select(
    [
        outcome.str.contains(r"pass", na=False),
        outcome.str.contains(r"fail", na=False)
    ],
    ["Approved", "Declined"],
    default="Other / Unclassified"
)

# ── Build heat-map data: Approved vs Declined by Municipality only ──
_group_cols = ["Bureau - Municipality"]

approved_counts = (
    df[df["Map Category"] == "Approved"]
    .groupby(_group_cols)
    .size()
    .reset_index(name="Count")
)
approved_counts["Category"] = "Approved"

declined_counts = (
    df[df["Map Category"] == "Declined"]
    .groupby(_group_cols)
    .size()
    .reset_index(name="Count")
)
declined_counts["Category"] = "Declined"

heatmap_data = pd.concat([approved_counts, declined_counts], ignore_index=True)

# Pivot: rows = municipality, columns = category
heatmap_pivot = heatmap_data.pivot_table(
    index="Bureau - Municipality",
    columns="Category",
    values="Count",
    aggfunc="sum",
    fill_value=0
)

# Ensure both categories appear in the right order
for col in ["Approved", "Declined"]:
    if col not in heatmap_pivot.columns:
        heatmap_pivot[col] = 0

heatmap_pivot = heatmap_pivot[["Approved", "Declined"]]
heatmap_pivot = heatmap_pivot.loc[
    heatmap_pivot.sum(axis=1).sort_values(ascending=False).index
]

# ── Heat map 1: Approved vs Declined by Municipality ──
fig = px.imshow(
    heatmap_pivot,
    labels={
        "x": "Category",
        "y": "Municipality",
        "color": "Number of Records"
    },
    text_auto=True,
    aspect="auto",
    color_continuous_scale="YlOrRd",
    title="Indigent Application Concentration by Municipality — Approved vs Declined"
)
fig.update_layout(height=500)
fig.show()

# ── Summary ──
print(f"\nTotal Vhembe records:  {len(df):,}")
print(f"Approved applicants:   {(df['Map Category'] == 'Approved').sum():,}")
print(f"Declined applicants:   {(df['Map Category'] == 'Declined').sum():,}")

<!doctype html>


Total Vhembe records:  141,395
Approved applicants:   93,181
Declined applicants:   48,214


In [0]:
# ── Export cleaned combined_df to CSV for Data Studio ──
# Applies ALL the same cleaning transformations as cell 5:
# 1. Drop redundant columns
# 2. Remove ALL control characters
# 3. Replace commas with spaces
# 4. Fill NULLs (0 for numeric, "Not Filled" for strings)
# 5. Shorten Rule values to Passed/Failed
# 6. Drop derived bucket columns
# 7. Drop non-dashboard columns
# 8. Save with csv.QUOTE_ALL for proper CSV parsing
import os
import csv
import re as _re

export_df = dashboard_df.copy()

# Keep Verification Risk Level in the export (set _drop_risk to True to exclude it)
_drop_risk = False
if _drop_risk and "Verification Risk Level" in export_df.columns:
    export_df = export_df.drop(columns=["Verification Risk Level"])

# ── Step 1: Drop redundant columns (same as cell 5) ──
cols_to_drop = [
    "Bureau - Mobile Phone 1", "Bureau - Mobile Phone 2", "Bureau - Mobile Phone 3",
    "Bureau - Work Phone 1", "Bureau - Work Phone 2", "Bureau - Work Phone 3",
    "Bureau - Home Phone 1", "Bureau - Home Phone 2", "Bureau - Home Phone 3",
    "Bureau - Has Email", "Bureau - Email 1", "Bureau - Email 2", "Bureau - Email 3",
    "Bureau - ResidentialAddress1Line1", "Bureau - ResidentialAddress1Line2",
    "Bureau - ResidentialAddress1Line3", "Bureau - ResidentialAddress1Line4",
    "Bureau - Has Mobile Phone", "Bureau - Contact Category",
    "Linked Spouse Application Record Key", "Bureau - Middle Name",
    "Municipal - Municipal Book", "Municipal - Account Number", "Municipal - Account Holder",
    "Municipal - IdNumber", "Municipal - Name", "Municipal - IDNumber",
    "Municipal - Town", "Municipal - MUNICIPALITY", "Municipal - TOWNSHIP_CODE",
    "Municipal - TOWNSHIP", "Municipal - ACCT_ID", "Municipal - BADGE_NBR",
    "Municipal - NAT_ID", "Municipal - NAME", "Municipal - ADDRESS1",
    "Municipal - ADDRESS2", "Municipal - ADDRESS3", "Municipal - TARIFF",
    "Municipal - AMP_LIMIT", "Municipal - FBE_TYPE", "Municipal - CLAIM_MONTH",
    "Municipal - Municipality", "Municipal - Township", "Municipal - AcountNo",
    "Municipal - IDNo", "Municipal - ApplicantName", "Municipal - Sheet",
    "Municipal - AppicantIdNumber", "Municipal - Notes"
]
cols_to_drop = [c for c in cols_to_drop if c in export_df.columns]
export_df = export_df.drop(columns=cols_to_drop)
print(f"Step 1: Dropped {len(cols_to_drop)} redundant columns — {len(export_df.columns)} remaining")

# ── Step 1b: Add Verification Risk Level (if not already present) ──
# Adapted for cleaned data: Rule values already shortened to Passed/Failed,
# EnrichmentOutcome and Bureau - Has Name already dropped.
# Uses Bureau - Age == 0 as proxy for missing bureau data (Unassessed).
if "Verification Risk Level" not in export_df.columns:
    import numpy as np

    _unassessed = pd.to_numeric(export_df["Bureau - Age"], errors="coerce").eq(0)

    _high = (
        (export_df["Rule - Valid ID Verification"].astype("string") == "Failed")
        | (export_df["Rule - Deceased Status"].astype("string") == "Failed")
        | (export_df["Rule - Multiple Accounts"].astype("string") == "Failed")
    )

    _gov_inc = (
        r"department|municipality|municipal|\bgovernment\b|provincial|"
        r"\bministry\b|national park|national health laboratory|"
        r"public hospital|public school|local government|"
        r"provincial admin|public works|water affairs|"
        r"national treasury|national biodiversity institute"
    )
    _gov_exc = (
        r"pension|pensioner|pension fund|\bgrant\b|sassa|"
        r"social security|old age|disability grant|child support"
    )
    _emp = export_df["Bureau - Employer"].astype("string").str.strip().str.casefold()
    _gov = (_emp.str.contains(_gov_inc, regex=True, na=False) & ~_emp.str.contains(_gov_exc, regex=True, na=False))

    _medium = (
        (export_df["Rule - Municipality Check"].astype("string") == "Failed")
        | (pd.to_numeric(export_df["Bureau - ResidentialAddress1PostalCode"], errors="coerce").eq(0))
        | (export_df["Rule - Monthly Income"].astype("string") == "Failed")
        | (export_df["Rule - Number Of Properties"].astype("string") == "Failed")
        | (pd.to_numeric(export_df["Bureau - Number Of Properties"], errors="coerce").ge(1))
        | (pd.to_numeric(export_df["Bureau - Number of Active Companies"], errors="coerce").ge(1))
        | (export_df["Rule - Directorship Status"].astype("string") == "Failed")
        | _gov
    )

    _approved = (export_df["Rule - Indigent Application"].astype("string") == "Passed")

    export_df["Verification Risk Level"] = np.select(
        [~_approved, _unassessed, _high, _medium],
        ["Not Applicable - Application Not Approved", "Unassessed - Insufficient information",
         "High - Urgent verification review", "Medium - Further assessment required"],
        default="Low - No selected risk flag identified",
    )
    print(f"Step 1b: Added Verification Risk Level - {len(export_df.columns)} columns total")
else:
    print(f"Step 1b: Verification Risk Level already present - keeping it")

# ── Step 2: Remove ALL control characters + commas ──
_ctrl_pattern = _re.compile('[' + ''.join(chr(i) for i in range(32)) + chr(127) + ']+')
for col in export_df.columns:
    s = export_df[col]
    if s.dtype == bool:
        s = s.astype("string")
    elif s.dtype == object:
        s = s.astype("string")
    else:
        continue
    s = s.str.replace(_ctrl_pattern, " ", regex=True)
    s = s.str.replace(",", " ", regex=False)
    s = s.str.replace(r' {2,}', ' ', regex=True)
    s = s.str.strip()
    export_df[col] = s
print(f"Step 2: Removed control characters and commas")

# ── Step 3: Fill NULLs ──
for col in export_df.columns:
    if export_df[col].dtype in ["int64", "float64", "long", "double"]:
        export_df[col] = export_df[col].fillna(0)
    elif export_df[col].dtype == "object" or export_df[col].dtype == "string":
        export_df[col] = export_df[col].fillna("Not Filled")
print(f"Step 3: Filled NULLs (0 for numeric, Not Filled for strings)")

# ── Step 4: Shorten Rule columns to Passed/Failed ──
rule_cols = [c for c in export_df.columns if c.startswith("Rule -")]
for col in rule_cols:
    export_df[col] = (
        export_df[col]
        .astype("string")
        .str.split(" - ")
        .str[0]
        .str.strip()
    )
if "Municipal - EnrichmentOutcome" in export_df.columns:
    export_df["Municipal - EnrichmentOutcome"] = (
        export_df["Municipal - EnrichmentOutcome"]
        .astype("string")
        .str.split(" - ")
        .str[-1]
        .str.strip()
    )
print(f"Step 4: Shortened {len(rule_cols)} Rule columns to Passed/Failed")

# ── Step 5: Drop derived bucket columns ──
bucket_cols = [
    "Bureau - IncomeBucket", "Bureau - DebtBucket",
    "Bureau - Directorship Bucket", "Bureau - Properties Bucket",
    "Bureau - Age Group"
]
bucket_cols = [c for c in bucket_cols if c in export_df.columns]
export_df = export_df.drop(columns=bucket_cols)
print(f"Step 5: Dropped {len(bucket_cols)} derived bucket columns — {len(export_df.columns)} remaining")

# ── Step 6: Drop non-dashboard columns ──
dashboard_drop = [
    "Municipal - Book", "Municipal - PhoneNumber", "Municipal - EnrichmentOutcome",
    "Bureau - IDN", "Bureau - First Name", "Bureau - Surname",
    "Bureau - Has Name", "Bureau - Deceased Status",
    "Bureau - Deceased Date", "Source_Sheet"
]
dashboard_drop = [c for c in dashboard_drop if c in export_df.columns]
export_df = export_df.drop(columns=dashboard_drop)
print(f"Step 6: Dropped {len(dashboard_drop)} non-dashboard columns — {len(export_df.columns)} remaining")

# ── Step 6b: Keep only Vhembe municipalities, replace others with "Other" ──
vhembe_municipalities = [
    "LIM341 - Musina",
    "LIM343 - Thulamela",
    "LIM344 - Makhado",
    "LIM345 - Collins Chabane"
]
export_df["Bureau - Municipality"] = (
    export_df["Bureau - Municipality"]
    .astype("string")
    .where(export_df["Bureau - Municipality"].isin(vhembe_municipalities), "Other")
)
print(f"Step 6b: Standardised municipalities to Vhembe + Other")

print(f"\nFinal table: {len(export_df.columns)} columns, {len(export_df):,} records")

# ── Save to CSV with full quoting ──
output_path = "/Workspace/Users/hlophetshepiso4@gmail.com/Vhembe District Municipality Data Analysis/VDM_Data_Studio_Export.csv"
export_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL, lineterminator="\n")

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"\nSaved: {output_path}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Total records: {len(export_df):,}")
print(f"Total columns: {len(export_df.columns)}")

Step 1: Dropped 0 redundant columns — 29 remaining
Step 1b: Added Verification Risk Level - 30 columns total
Step 2: Removed control characters and commas
Step 3: Filled NULLs (0 for numeric, Not Filled for strings)
Step 4: Shortened 11 Rule columns to Passed/Failed
Step 5: Dropped 0 derived bucket columns — 30 remaining
Step 6: Dropped 0 non-dashboard columns — 30 remaining
Step 6b: Standardised municipalities to Vhembe + Other

Final table: 30 columns, 170,091 records

Saved: /Workspace/Users/hlophetshepiso4@gmail.com/Vhembe District Municipality Data Analysis/VDM_Data_Studio_Export.csv
File size: 53.41 MB
Total records: 170,091
Total columns: 30
